# Milestone 5: Music Maven API Integration
Liam

Objective 2, finished up to PI.3

In [1]:
import json
import threading
import time
import numpy as np
import pandas as pd
import requests
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

In [2]:
MIR_FEATURES = ["tempo", "energy", "valence", "danceability", "key", "mode"]

profiles = pd.read_csv("artist_profiles.csv")

## Shared Utilities
Normalization and distance functions CARRIED OVER VIA M3/4. (KEVINS WORK)

In [3]:
def normalize_features(X, method="minmax"):
    X = np.array(X, dtype=float)
    if method == "minmax":
        mins, maxs = X.min(axis=0), X.max(axis=0)
        ranges = maxs - mins
        with np.errstate(invalid="ignore", divide="ignore"):
            X_norm = np.where(ranges == 0, 0.0, (X - mins) / ranges)
        return X_norm, {"method": "minmax", "mins": mins, "maxs": maxs}
    means, stds = X.mean(axis=0), X.std(axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        X_norm = np.where(stds == 0, 0.0, (X - means) / stds)
    return X_norm, {"method": "zscore", "means": means, "stds": stds}

def apply_normalization(x, params):
    x = np.array(x, dtype=float)
    if params["method"] == "minmax":
        ranges = params["maxs"] - params["mins"]
        with np.errstate(invalid="ignore", divide="ignore"):
            return np.where(ranges == 0, 0.0, (x - params["mins"]) / ranges)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(params["stds"] == 0, 0.0, (x - params["means"]) / params["stds"])

def prepare_weights(weights, n):
    if weights is None:
        return np.full(n, 1.0 / n)
    w = np.array(weights, dtype=float)
    return w / w.sum()

def weighted_euclidean_batch(query, candidates, weights=None):
    query = np.array(query, dtype=float)
    candidates = np.array(candidates, dtype=float)
    w = prepare_weights(weights, query.shape[0])
    diff = candidates - query
    return np.sqrt((w * diff * diff).sum(axis=1))

In [4]:
class SimilarityEngine:
    def __init__(self, profiles_df, normalize="minmax"):
        self._df = profiles_df.reset_index(drop=True)
        raw = self._df[MIR_FEATURES].to_numpy(dtype=float)
        self._norm_matrix, self._norm_params = normalize_features(raw, method=normalize)

    def search(self, artist_id, k=10, weights=None, exclude_genres=None,
               exclude_year_range=None, require_genres=None):
        query_idx = int((self._df["artist_id"] == artist_id).idxmax())
        candidates = self._apply_filters(artist_id, exclude_genres, exclude_year_range, require_genres)
        weight_vec = np.array([weights.get(f, 0.0) for f in MIR_FEATURES]) if weights else None
        cand_matrix = self._norm_matrix[candidates.index.tolist()]
        dists = weighted_euclidean_batch(self._norm_matrix[query_idx], cand_matrix, weight_vec)
        order = np.argsort(dists)[:min(k, len(dists))]
        result = candidates.iloc[order][["artist_id", "artist_name"]].copy().reset_index(drop=True)
        result["distance"] = dists[order]
        result["rank"] = np.arange(1, len(order) + 1)
        return result

    def _apply_filters(self, artist_id, exclude_genres, exclude_year_range, require_genres):
        df = self._df[self._df["artist_id"] != artist_id].copy()
        if exclude_genres:
            excl = {g.lower() for g in exclude_genres}
            df = df[~df["genres"].apply(lambda g: bool({x.strip().lower() for x in str(g).split(",")} & excl))]
        if exclude_year_range:
            s, e = exclude_year_range
            df = df[~((df["year_min"] <= e) & (df["year_max"] >= s))]
        if require_genres:
            req = {g.lower() for g in require_genres}
            df = df[df["genres"].apply(lambda g: bool({x.strip().lower() for x in str(g).split(",")} & req))]
        return df


class ArtistKNNClassifier:
    def __init__(self, profiles_df, k=5, weights=None, normalize="minmax"):
        self.k = k
        self._df = profiles_df.reset_index(drop=True)
        self._weight_vec = np.array([weights.get(f, 0.0) for f in MIR_FEATURES]) if weights else None
        raw = self._df[MIR_FEATURES].to_numpy(dtype=float)
        self._norm_matrix, self._norm_params = normalize_features(raw, method=normalize)
        self._artist_ids   = self._df["artist_id"].to_numpy()
        self._artist_names = self._df["artist_name"].to_numpy()

    def classify(self, song_features, k=None):
        k = k or self.k
        raw_vec = np.array([float(song_features[f]) for f in MIR_FEATURES])
        query_vec = apply_normalization(raw_vec, self._norm_params)
        dists = weighted_euclidean_batch(query_vec, self._norm_matrix, self._weight_vec)
        nn_idx = np.argsort(dists)[:k]
        artist_data = {}
        for aid, aname, dist in zip(self._artist_ids[nn_idx], self._artist_names[nn_idx], dists[nn_idx]):
            if aid not in artist_data:
                artist_data[aid] = {"artist_id": aid, "artist_name": aname, "votes": 0, "total_dist": 0.0}
            artist_data[aid]["votes"] += 1
            artist_data[aid]["total_dist"] += dist
        results = [{**e, "probability": e["votes"] / k, "avg_distance": e["total_dist"] / e["votes"]}
                   for e in artist_data.values()]
        results.sort(key=lambda x: (-x["votes"], x["avg_distance"]))
        return results

## PI.2: JSON Query Schemas
Pydantic models define the contract for both endpoints.

In [5]:
class SimilarityQuery(BaseModel):
    artist_id:          str
    k:                  int                        = Field(default=10, ge=1, le=50)
    weights:            Optional[dict[str, float]] = None
    exclude_genres:     Optional[list[str]]        = None
    require_genres:     Optional[list[str]]        = None
    exclude_year_range: Optional[tuple[int, int]]  = None


class ClassifyQuery(BaseModel):
    features: dict[str, float]
    k:        int                        = Field(default=5, ge=1, le=50)
    weights:  Optional[dict[str, float]] = None


print(json.dumps(SimilarityQuery.model_json_schema(), indent=2))
print(json.dumps(ClassifyQuery.model_json_schema(), indent=2))

{
  "properties": {
    "artist_id": {
      "title": "Artist Id",
      "type": "string"
    },
    "k": {
      "default": 10,
      "maximum": 50,
      "minimum": 1,
      "title": "K",
      "type": "integer"
    },
    "weights": {
      "anyOf": [
        {
          "additionalProperties": {
            "type": "number"
          },
          "type": "object"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Weights"
    },
    "exclude_genres": {
      "anyOf": [
        {
          "items": {
            "type": "string"
          },
          "type": "array"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Exclude Genres"
    },
    "require_genres": {
      "anyOf": [
        {
          "items": {
            "type": "string"
          },
          "type": "array"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": 

## PI.1: FastAPI Microservice
Two endpoints backed directly by the Milestone 3/4 engines.
Uvicorn runs in a background thread so the notebook stays interactive.

In [6]:
API_PORT = 8766

_engine     = SimilarityEngine(profiles)
_classifier = ArtistKNNClassifier(profiles)

app = FastAPI(title="Music Maven Similarity API")


@app.get("/health")
def health():
    return {"status": "ok", "artists": len(profiles)}


@app.post("/similarity")
def similarity(query: SimilarityQuery):
    result_df = _engine.search(
        artist_id          = query.artist_id,
        k                  = query.k,
        weights            = query.weights,
        exclude_genres     = query.exclude_genres,
        require_genres     = query.require_genres,
        exclude_year_range = query.exclude_year_range,
    )
    return {"query": query.model_dump(), "results": result_df.to_dict(orient="records")}


@app.post("/classify")
def classify(query: ClassifyQuery):
    clf = ArtistKNNClassifier(profiles, k=query.k, weights=query.weights)
    predictions = clf.classify(query.features)
    return {"query": query.model_dump(), "predictions": predictions}


threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=API_PORT, log_level="warning"), daemon=True).start()
time.sleep(1.5)

print(requests.get(f"http://127.0.0.1:{API_PORT}/health").json())

{'status': 'ok', 'artists': 9382}


## PI.3: Query Parser
Validates raw JSON and dispatches to the correct endpoint. Intended as the interface layer between any upstream caller and the API.

In [7]:
API_BASE = f"http://127.0.0.1:{API_PORT}"

def parse_and_dispatch(raw_json: str) -> dict:
    """
    Validate raw JSON and dispatch to the right endpoint.
    Expects a top-level `endpoint` key ("similarity" | "classify"),
    all remaining keys forwarded as the query body.
    """
    data     = json.loads(raw_json)
    endpoint = data.pop("endpoint")
    schema   = SimilarityQuery if endpoint == "similarity" else ClassifyQuery
    schema.model_validate(data)  # raises ValidationError on bad input
    resp = requests.post(f"{API_BASE}/{endpoint}", json=data, timeout=10)
    return resp.json()

In [8]:
# similarity query
r = parse_and_dispatch(json.dumps({
    "endpoint":  "similarity",
    "artist_id": profiles["artist_id"].iloc[0],
    "k": 5,
    "weights": {"energy": 0.4, "danceability": 0.4, "tempo": 0.1, "valence": 0.1},
}))
pd.DataFrame(r["results"])[["rank", "artist_id", "artist_name", "distance"]]

,rank,artist_id,artist_name,distance
0,1,A07656,Koffee,0.014714
1,2,A01587,Biel,0.025377
2,3,A15806,Your Smith,0.025502
3,4,A04630,Fabolous,0.028141
4,5,A10052,New Hope Club,0.029556


In [9]:
# classify query
sample_features = {f: float(profiles[f].iloc[0]) for f in MIR_FEATURES}
r2 = parse_and_dispatch(json.dumps({"endpoint": "classify", "features": sample_features, "k": 5}))
pd.DataFrame(r2["predictions"])[["artist_id", "artist_name", "votes", "probability"]]

,artist_id,artist_name,votes,probability
0,A00001,#TocoParaVos,1,0.2
1,A11998,Sage the Gemini,1,0.2
2,A07215,Jurassic 5,1,0.2
3,A13109,Suchmos,1,0.2
4,A02312,Cameo,1,0.2


## Summary

| PI |  Comments |
|---|---|
| PI.1 | FastAPI app with `/health`, `/similarity`, `/classify`; uvicorn in background thread |
| PI.2 | `SimilarityQuery` and `ClassifyQuery` Pydantic models with field validation |
| PI.3 | `parse_and_dispatch()` validates, routes, and forwards JSON to the correct endpoint |